# Logistics Analytics and Shipment Delay Modeling

**Author: Fairouz Juma**

Academic project: clean shipment data, investigate delays, compare Spark ML regressors, and communicate findings in Power BI. See the repository README for setup, original results and limitations.

The default input is a generated demonstration dataset. Set `SHIPMENTS_CSV` to your original CSV to reproduce the full analysis. Demo results are not the original project results.

## 1. Imports and environment

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator


## 2. Load shipment data

In [ ]:
spark = SparkSession.builder.appName("TransLogix").getOrCreate() # create spark session
from pathlib import Path
import os
ROOT = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent
path = os.environ.get("SHIPMENTS_CSV", str(ROOT / "data" / "shipments_demo.csv"))
if not Path(path).is_file():
    raise FileNotFoundError(f"Dataset not found: {path}. See data/README.md.")

## 3. Inspect schema

In [ ]:
df = (spark.read
      .option("header", "true")
      .option("inferSchema", "true")
      .csv(path))

## 4. Missing values and duplicate IDs

In [ ]:
print("Rows:", df.count(), "Cols:", len(df.columns))
df.show(5)

In [ ]:
df.printSchema()

## 5. Clean records

In [ ]:
total_rows = df.count()
print("total rows", total_rows)
print("Missing in column")

for column_name in df.columns:
    missing_count=df.filter(F.col(column_name).isNull()).count()

    if missing_count>0:
        print(column_name,':' , missing_count)

In [ ]:
unique_ids=df.select("shipment_id").distinct().count()
duplicate_count=total_rows-unique_ids
print("Duplicate shipment_ids:",duplicate_count)

In [ ]:
df_clean = df.dropna(subset=["delay_minutes","actual_departure","actual_arrival"]) #remove rows with nulls
print("Rows after removing nulls:",df_clean.count())

## 6. Derive delay categories and transit times

In [ ]:
df_clean = df_clean.fillna({"fragile": "No"}) # fill nulls in fragile col with 'No'
avg_weight = df_clean.select(F.avg("product_weight_kg")).first()[0]
df_clean = df_clean.fillna({"product_weight_kg": avg_weight}) # fill product weight with avg

In [ ]:
date_cols = ["scheduled_departure","scheduled_arrival","actual_departure","actual_arrival"]

for c in date_cols:
    df_clean = df_clean.withColumn(c, F.trim(F.col(c))) #clean spaces
    df_clean = df_clean.withColumn(c,F.coalesce(F.expr(f"try_to_timestamp({c}, 'M/d/yyyy HH:mm')"),
                                                F.expr(f"try_to_timestamp({c}, 'M/d/yyyy H:mm')"),
                                                F.expr(f"try_to_timestamp({c}, 'M/d/yyyy HH:mm:ss')"),
                                                F.expr(f"try_to_timestamp({c}, 'M/d/yyyy H:mm:ss')")))
# convert string values to timestamp using different formats

print("Rows after time stamp parsing:", df_clean.count())
df_clean.select(date_cols).show(5, truncate=False)


## 7. Exploratory analysis

In [ ]:
#create delay categories to group shipments by their delay severity
df_clean = df_clean.withColumn("delay_degree",
                               F.when(F.col("delay_minutes")<0,"In advance")
                               .when(F.col("delay_minutes")==0,"On time")
                               .when(F.col("delay_minutes")<=30,"Minor delay")
                               .when(F.col("delay_minutes")<=60,"Moderate delay")
                               .otherwise("Severe Delay"))

In [ ]:
#calculate trasit time in hours
df_clean = df_clean.withColumn("actual_transit_hours",
 (F.unix_timestamp("actual_arrival") - F.unix_timestamp("actual_departure"))/3600)

In [ ]:
num_cols = ["delay_minutes","distance_km","planned_transit_hours","origin_congestion_idx",
            "dest_congestion_idx","vehicle_reliability","product_weight_kg"]
df_clean.select(num_cols).describe().show() #descriptive statistics of numerical cols

In [ ]:
#calculate min max and avg delay statistics
df_clean.select(F.count("*").alias("rows"),
                F.min("delay_minutes"),
                F.max("delay_minutes"),
                F.avg("delay_minutes")).show()

df_clean.groupBy("delay_degree").count().show() #count shipment of each category delay

In [ ]:
#summary of shipments and delays by origin hub
hub_delay = (df_clean
             .groupBy("origin_hub", "origin_hub_name", "origin_region", "origin_country")
             .agg(F.count("*").alias("shipments"),
                  F.round(F.avg("delay_minutes")).alias("avg_delay_min"), #avg delay for shipments from each hub
                  F.round(F.max("delay_minutes")).alias("max_delay_min")  #max delay for each hub
                  ))
hub_delay.show(truncate=False)

In [ ]:
#summary of shipments and delays by destination hub
dest_hub_delay = (df_clean
                  .groupBy("destination_hub", "destination_hub_name", "destination_region", "destination_country")
                  .agg(F.count("*").alias("shipments"),
                       F.round(F.avg("delay_minutes")).alias("avg_delay_min"),
                       F.round(F.max("delay_minutes")).alias("max_delay_min")))
dest_hub_delay.show(truncate=False)

In [ ]:
#analyze delays by route and transport mode
route_delay = (df_clean
               .groupBy("route_id", "route_name", "origin_hub", "destination_hub", "transport_mode")
               .agg(F.count("*").alias("shipments"),
                    F.round(F.avg("delay_minutes")).alias("avg_delay_min"),
                    F.round(F.avg("distance_km")).alias("avg_distance_km"),
                    F.round(F.avg("planned_transit_hours")).alias("avg_planned_hours"),
                    F.round(F.avg("actual_transit_hours")).alias("avg_actual_hours") )
               .withColumn("extra_transit_hours", F.round(F.col("avg_actual_hours") - F.col("avg_planned_hours")))
               .orderBy(F.col("avg_delay_min").desc()))

route_delay.show(20, truncate=False)

In [ ]:
#create a shipment date column from scheduled departure column
delay_by_day = (df_clean
                .withColumn("ship_date",F.to_date("scheduled_departure"))
                .groupBy("ship_date")
                .agg(F.count("*").alias("shipments"),
                     F.round(F.avg("delay_minutes")).alias("avg_delay_min"))
                .orderBy(F.col("avg_delay_min").desc()))
delay_by_day.show(truncate=False) #show the days w/ highest avg delays

In [ ]:
delay_by_day_ranked = delay_by_day.orderBy(F.col("shipments").desc())
delay_by_day_ranked.show() #show most busy days and avg delays

In [ ]:
#group shipments by their origin congestion levels
congestion_effect = (df_clean
                     .select("delay_minutes", "origin_congestion_idx")
                     .groupBy(F.round("origin_congestion_idx",1).alias("origin_cong_bucket"))
                     .agg(F.count("*").alias("shipments"),
                          F.round(F.avg("delay_minutes")).alias("avg_delay_min"))
                     .orderBy("origin_cong_bucket"))
congestion_effect.show()

In [ ]:
#group shipments by their destination congestion level
congestion_effect = (df_clean
                     .select("delay_minutes", "dest_congestion_idx")
                     .groupBy(F.round("dest_congestion_idx",1).alias("dest_cong_bucket"))
                     .agg(F.count("*").alias("shipments"),
                          F.round(F.avg("delay_minutes")).alias("avg_delay_min"))
                     .orderBy("dest_cong_bucket"))
congestion_effect.show()

In [ ]:
#comparison of delays for fragile/non fragile shipments
fragile_effect = (df_clean
                  .groupBy("fragile")
                  .agg(F.count("*").alias("shipments"),
                       F.round(F.avg("delay_minutes")).alias("avg_delay_min"),
                       F.round(F.max("delay_minutes")).alias("max_delay_min"))
                  .orderBy(F.col("avg_delay_min").desc()))
fragile_effect.show(truncate=False)

## 8. Export cleaned data for Power BI

In [ ]:
weight_delay = (df_clean
                .groupBy(F.round("product_weight_kg").alias("weight_kg"))
                .agg(F.count("*").alias("shipments"),
                     F.round(F.avg("delay_minutes")).alias("avg_delay_min")))

weight_delay.orderBy(F.col("weight_kg").desc()).show(20, truncate=False)
weight_delay.orderBy(F.col("weight_kg").asc()).show(20, truncate=False)

## 9. Prepare regression features

In [ ]:
#save the new data for further visualization
output_path = str(ROOT / "outputs" / "cleaned_shipments")
df_clean.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_path)

In [ ]:
model_df = df_clean.select("delay_minutes","distance_km","planned_transit_hours",
                           "origin_congestion_idx","dest_congestion_idx", "vehicle_reliability",
                           "transport_mode", "origin_hub", "destination_hub", "route_id","fragile")
model_df.count()

In [ ]:
cat_cols = ["transport_mode", "origin_hub", "destination_hub","route_id","fragile"]
num_cols = ["distance_km", "planned_transit_hours", "origin_congestion_idx",
            "dest_congestion_idx", "vehicle_reliability"]

indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep") for c in cat_cols]

encoder = OneHotEncoder(inputCols=[c+"_idx" for c in cat_cols],outputCols=[c+"_ohe" for c in cat_cols])

assembler = VectorAssembler(inputCols=num_cols + [c+"_ohe" for c in cat_cols],outputCol="features")

## 10. Train and compare models

In [ ]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)

In [ ]:
lr = LinearRegression(featuresCol="features",labelCol="delay_minutes")

lr_pipe = Pipeline(stages=indexers +[encoder,assembler,lr])
lr_model= lr_pipe.fit(train_df)
lr_pred= lr_model.transform(test_df)

In [ ]:
rf = RandomForestRegressor(featuresCol ="features", labelCol= "delay_minutes", numTrees=50, seed=42)

rf_pipe= Pipeline(stages=indexers +[encoder,assembler,rf])
rf_model= rf_pipe.fit(train_df)
rf_pred= rf_model.transform(test_df)

In [ ]:
gbt =GBTRegressor(featuresCol= "features", labelCol ="delay_minutes", maxIter=50, seed=42)

gbt_pipe= Pipeline(stages=indexers +[encoder,assembler,gbt])
gbt_model= gbt_pipe.fit(train_df)
gbt_pred= gbt_model.transform(test_df)

In [ ]:
evaluator_rmse = RegressionEvaluator(labelCol="delay_minutes",predictionCol="prediction",metricName="rmse")
evaluator_r2= RegressionEvaluator(labelCol="delay_minutes", predictionCol="prediction", metricName="r2")

In [ ]:
models = ["Linear Regression","Random Forest","Gradient Boosted Trees"]
rmse_values= [evaluator_rmse.evaluate(lr_pred),evaluator_rmse.evaluate(rf_pred),evaluator_rmse.evaluate(gbt_pred)]
r2_values= [evaluator_r2.evaluate(lr_pred),evaluator_r2.evaluate(rf_pred),evaluator_r2.evaluate(gbt_pred)]

for v in zip(models,rmse_values,r2_values):
    print(v)

## Interpretation and next steps

The original saved run has modest R-squared values (~0.24-0.25); these are baseline models, not production forecasts. Use a chronological holdout and cross-validation before deployment. Check nulls after timestamp parsing, validate feature availability at prediction time, and consider additional operational predictors. `coalesce(1)` simplifies the Power BI export but is unsuitable for very large distributed exports.